In [ ]:
# 1. Cài đặt môi trường
!pip uninstall -y transformers accelerate -q
!pip install -q transformers accelerate datasets scikit-learn

import os
import torch
import time
import numpy as np
from PIL import Image
from datasets import load_dataset
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

# ==========================================
# CẤU HÌNH & KẾT NỐI
# ==========================================
MODEL_NAME = "microsoft/resnet-18"
from google.colab import drive
drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/MFND_Baseline_ResNet18"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Tải dữ liệu và Tiền xử lý Hình ảnh
print("⏳ Loading dataset...")
dataset = load_dataset("anson-huang/mirage-news")
image_processor = AutoImageProcessor.from_pretrained(MODEL_NAME)

def preprocess(example):
    try:
        image = example["image"]
        # Đảm bảo ảnh ở định dạng RGB
        if image is None or not isinstance(image, Image.Image):
            image = Image.new("RGB", (224, 224), (0, 0, 0))
        elif image.mode != "RGB":
            image = image.convert("RGB")

        inputs = image_processor(image, return_tensors="pt")
        return {
            "pixel_values": inputs["pixel_values"][0],
            "labels": example["label"]
        }
    except Exception:
        return None

print("🖼️ Mapping images (this may take a while)...")
encoded_dataset = dataset.map(preprocess, remove_columns=dataset["train"].column_names)
encoded_dataset = encoded_dataset.filter(lambda x: x is not None)

# ==========================================
# 3. ĐO LƯỜNG THAM SỐ (PARAMS)
# ==========================================
model = AutoModelForImageClassification.from_pretrained(
    MODEL_NAME, num_labels=2, ignore_mismatched_sizes=True
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n{'='*30}")
print(f"📊 Total Params: {total_params:,}")
print(f"📊 Trainable Params: {trainable_params:,}")
print(f"{'='*30}\n")

# 4. Hàm tính chỉ số
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds = np.argmax(probs, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": accuracy_score(labels, preds), "f1": f1, "auc": roc_auc_score(labels, probs[:, 1])}

# 5. Cấu hình Huấn luyện
training_args = TrainingArguments(
    output_dir=SAVE_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=5,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# ==========================================
# 6. HUẤN LUYỆN & ĐO LƯỜNG VRAM/TIME
# ==========================================
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

print("🚀 Starting ResNet-18 Fine-tuning...")
start_train_time = time.time()
trainer.train()
end_train_time = time.time()

training_time_sec = end_train_time - start_train_time
vram_used_mb = torch.cuda.max_memory_reserved() / (1024 ** 2)

print(f"\n{'='*30}")
print(f"⏱️ Training Time: {training_time_sec/60:.2f} minutes")
print(f"💾 VRAM Used: {vram_used_mb:.2f} MB")
print(f"{'='*30}\n")

# ==========================================
# 7. ĐO LƯỜNG ĐỘ TRỄ (LATENCY)
# ==========================================
model.eval()
dummy_image = Image.new("RGB", (224, 224), (128, 128, 128))
inputs = image_processor(dummy_image, return_tensors="pt").to(device)

for _ in range(10): # Warm-up
    with torch.no_grad():
        _ = model(**inputs)

start_lat = time.time()
for _ in range(100):
    with torch.no_grad():
        _ = model(**inputs)
end_lat = time.time()

latency_ms = ((end_lat - start_lat) / 100) * 1000
print(f"⚡ Inference Latency: {latency_ms:.2f} ms/sample\n")

# ==========================================
# 8. ĐÁNH GIÁ CHI TIẾT CÁC BỘ TEST
# ==========================================
test_splits = ["test1_nyt_mj", "test2_bbc_dalle", "test3_cnn_dalle", "test4_bbc_sdxl", "test5_cnn_sdxl"]
results_storage = {}

for split in test_splits:
    out = trainer.predict(encoded_dataset[split])
    acc = out.metrics.get("test_accuracy", 0)
    f1 = out.metrics.get("test_f1", 0)
    auc = out.metrics.get("test_auc", 0)
    results_storage[split] = {"acc": acc, "f1": f1, "auc": auc}
    print(f"✅ {split}: Acc={acc*100:.2f}% | F1={f1*100:.2f}% | AUC={auc:.4f}")

# Tổng hợp kết quả
in_domain = results_storage["test1_nyt_mj"]
ood_vals = [results_storage[s] for s in test_splits[1:]]

print(f"\n📍 In-domain  | Acc: {in_domain['acc']*100:.2f}% | F1: {in_domain['f1']*100:.2f}% | AUC: {in_domain['auc']:.4f}")
print(f"🌐 Avg Out-domain | Acc: {np.mean([x['acc'] for x in ood_vals])*100:.2f}% | F1: {np.mean([x['f1'] for x in ood_vals])*100:.2f}% | AUC: {np.mean([x['auc'] for x in ood_vals]):.4f}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 130.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 39.8 MB/s eta 0:00:00
Mounted at /content/drive
⏳ Loading dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/655M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/143M [00:00<?, ?B/s]

...

✅ test1_nyt_mj: Acc=98.00% | F1=97.98% | AUC=0.9969


✅ test2_bbc_dalle: Acc=70.20% | F1=65.11% | AUC=0.7985


✅ test3_cnn_dalle: Acc=70.80% | F1=65.07% | AUC=0.7911


✅ test4_bbc_sdxl: Acc=71.60% | F1=67.87% | AUC=0.8194


✅ test5_cnn_sdxl: Acc=76.00% | F1=72.97% | AUC=0.8547

📍 In-domain  | Acc: 98.00% | F1: 97.98% | AUC: 0.9969
🌐 Avg Out-domain | Acc: 72.15% | F1: 67.76% | AUC: 0.8159
